## What is expierment tracking?

machine learning and deep learning are higly expiermental 


You have to put on your artist's beret/chef's hat to cook up lots of different models.

And you have to put on your scientist's coat to track the results of various combinations of data, model architectures and training regimes.

That's where experiment tracking comes in.

If you're running lots of different experiments, experiment tracking helps you figure out what works and what doesn't.




##  Different ways to track machine learning expierments
    

| Method | Setup | Pros | Cons | Cost |
| :--- | :--- | :--- | :--- | :--- |
| Python dictionaries, CSV files, print outs | None | Easy to setup, runs in pure Python | Hard to keep track of large numbers of experiments | Free |
| [TensorBoard](https://www.tensorflow.org/tensorboard) | Minimal, install `tensorboard` | Extensions built into PyTorch, widely recognized and used, easily scales. | User-experience not as nice as other options. | Free |
| [Weights & Biases Experiment Tracking](https://wandb.ai/) | Minimal, install `wandb`, make an account | Incredible user experience, make experiments public, tracks almost anything. | Requires external resource outside of PyTorch. | Free for personal use |
| [MLFlow](https://mlflow.org/) | Minimal, install `mlflow` and start tracking | Fully open-source MLOps lifecycle management, many integrations. | Little bit harder to setup a remote tracking server than other services. | Free |

### 0.Getting setup

In [2]:
import sys
from pathlib import Path
# Add the parent directory and going_modular_05 to sys.path to allow imports
sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent / "going_modular_05"))
from going_modular_05 import going_modular
from going_modular import data_setup,engine
from torchvision import transforms
from torch import nn
import matplotlib.pyplot as plt
import torch



In [3]:
device = "cuda" if torch.cuda.is_available() else " cpu"  
device

'cuda'

Creating a helper function to set seeds

In [4]:
def set_seeds(seed : int = 42):
    """ Set random sets for torch operations    

    Args:
        seed(int,optional) : Random seed to set.Defaults to 42.
    """
    #Set the seed for general torch operations
    torch.manual_seed(42)
    torch.cuda.manual_seed(42)
    

### 1.Getting data (Making a `download_data function`)

In [5]:
import os
import zipfile

from pathlib import Path

import requests

def download_data(source: str, 
                  destination: str,
                  remove_source: bool = True) -> Path:
    """Downloads a zipped dataset from source and unzips to destination.

    Args:
        source (str): A link to a zipped file containing data.
        destination (str): A target directory to unzip data to.
        remove_source (bool): Whether to remove the source after downloading and extracting.
    
    Returns:
        pathlib.Path to downloaded data.
    
    Example usage:
        download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                      destination="pizza_steak_sushi")
    """
    # Setup path to data folder
    data_path = Path("data/")
    image_path = data_path / destination

    # If the image folder doesn't exist, download it and prepare it... 
    if image_path.is_dir():
        print(f"[INFO] {image_path} directory exists, skipping download.")
    else:
        print(f"[INFO] Did not find {image_path} directory, creating one...")
        image_path.mkdir(parents=True, exist_ok=True)
        
        # Download pizza, steak, sushi data
        target_file = Path(source).name
        with open(data_path / target_file, "wb") as f:
            request = requests.get(source)
            print(f"[INFO] Downloading {target_file} from {source}...")
            f.write(request.content)

        # Unzip pizza, steak, sushi data
        with zipfile.ZipFile(data_path / target_file, "r") as zip_ref:
            print(f"[INFO] Unzipping {target_file} data...") 
            zip_ref.extractall(image_path)

        # Remove .zip file
        if remove_source:
            os.remove(data_path / target_file)
    
    return image_path

image_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                           destination="pizza_steak_sushi")
image_path

[INFO] data\pizza_steak_sushi directory exists, skipping download.


WindowsPath('data/pizza_steak_sushi')

### 2.Create Dataset and DataLoaders

**To transform our images into tensors, we can use:**

1.Manually created transforms using `torchvision.transforms.`

2.Automatically created transforms using `torchvision.models.MODEL_NAME.MODEL_WEIGHTS.DEFAULT.transforms()`

### 2.1 Create Dataloaders using manually created transforms

In [9]:
from going_modular import data_setup

In [11]:
#Test dir
train_dir = image_path/"train"
test_dir = image_path/"test"

#Setup imagenet normalization levels
normalize = transforms.Normalize(mean = [0.458,0.456,0.406],
                                std = [0.299,0.224,0.225])

#Creating transforms pipeline
manual_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    normalize
])

print(f"Manaully created transforms : {manual_transform}")

train_dataloader , test_dataloader,class_names,class_dict = data_setup.create_dataloaders(
train_dir = train_dir,test_dir = test_dir,train_transform = manual_transform,test_transform = manual_transform,
batch_size = 32
)

train_dataloader,test_dataloader,class_names,class_dict
                                

Manaully created transforms : Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.458, 0.456, 0.406], std=[0.299, 0.224, 0.225])
)


(<torch.utils.data.dataloader.DataLoader at 0x233a324e170>,
 ['pizza', 'steak', 'sushi'],
 {'pizza': 0, 'steak': 1, 'sushi': 2})

### 2.2 Create dataloaders using automatically created transforms